# Setup Postgres

In [1]:
#Set up Postgres Connection
#psycopg2 is the PostgreSQL driver for
import psycopg2
from psycopg2 import sql

# Database connection details

host = "localhost"  # Your PostgreSQL host
user = "postgres"  # Your PostgreSQL username
password = "admin"  # Your PostgreSQL password
port = "5432"  # Default port for PostgreSQL
new_database = "nhl"  # Name of the new database

# Connect to the PostgreSQL server and create a new database if needed
try:
    conn = psycopg2.connect(dbname="postgres", user=user, password=password, host=host, port=port)
    conn.autocommit = True  # Enable autocommit for database creation

    # Create a cursor
    cursor = conn.cursor()

    # Check if the database already exists
    cursor.execute(sql.SQL("SELECT 1 FROM pg_database WHERE datname = %s"), [new_database])
    exists = cursor.fetchone()

    if not exists:
        cursor.execute(sql.SQL("CREATE DATABASE {}").format(sql.Identifier(new_database)))
        print(f"Database '{new_database}' created successfully.")
    else:
        print(f"Database '{new_database}' already exists.")
    
    # Close the cursor and connection
    cursor.close()
    conn.close()

except Exception as e:
    print(f"Error: {e}")

Database 'nhl' created successfully.


### Create Database in PostgreSQL

In [4]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError

# Define your PostgreSQL connection parameters
username = 'postgres'  # PostgreSQL username
password = 'admin'  # PostgreSQL password
host = 'localhost'  # PostgreSQL host
port = '5432'  # Default PostgreSQL port
database_name = 'nhl'  # Your PostgreSQL database name

# Create the connection string to PostgreSQL
connection_string = f'postgresql+psycopg2://{username}:{password}@{host}:{port}/{database_name}'

# Establish the connection to PostgreSQL
engine = create_engine(connection_string)

# Define table creation SQL statements
commands = [
    # Table 1 : team_info
    """
    CREATE TABLE team_info (
        team_id VARCHAR(3) PRIMARY KEY,             -- Unique identifier for each team (Primary Key)
        franchise_id INT,                   -- Franchise identifier for each team
        short_name VARCHAR(15),             -- Short name for the team (location)
        team_name VARCHAR(15) ,     -- Full name of the team
        abbreviation VARCHAR(10)     -- Abbreviation for the team (e.g., NJD, BOS)
    );
    """,

   
    # Table 2 : player_info
    """
    CREATE TABLE player_info (
        player_id VARCHAR(10) PRIMARY KEY,         -- Unique identifier for each player (NOT NULL by default as it's a primary key)
        first_name VARCHAR(20) ,           -- Player's first name (Required)
        last_name VARCHAR(20),            -- Player's last name (Required)
        nationality VARCHAR(3),                    -- Player's nationality in abbreviated shortform
        birth_city VARCHAR(30),
    	primary_position VARCHAR(3) ,      -- Player's primary position (R-ight Wing/L-eft Wing/C-enter/D-efence/G-oalie)
        birth_date DATE ,                  -- Player's date of birth (Required)
        birth_state_province VARCHAR(10),
    	height_cm FLOAT,                    -- Player's height in cm (Required)
        weight_kg FLOAT,                     -- Player's weight in kg (Required)
    	shoots_catches VARCHAR(5)                 -- Shooting/Catching style (L-eft or R-ight Handed)
    );
    """,

    # Table 3: game
    """
    CREATE TABLE game (
        game_id VARCHAR(12) PRIMARY KEY,
        season INT,
        game_type CHAR(1), -- Example: 'R' for Regular, 'P' for Playoffs
        date_time_gmt TIMESTAMP,
        away_team_id VARCHAR(3),
        home_team_id VARCHAR(3),
        away_goals INT,
        home_goals INT,
        home_rink_side_start VARCHAR(5),
        venue VARCHAR(50) , -- Example: "United Center"
        venue_time_zone_id VARCHAR(20),
        venue_time_zone_offset INT , -- Example: -5
        venue_time_zone_tz VARCHAR(3),
        hoa VARCHAR(4) , -- "home" or "away"
        settled_in VARCHAR(10)  -- Example: "REG", "OT", "SO"
    );
    """,
    
   
    # Table 4: game_plays
    """
    CREATE TABLE game_plays (
        play_id VARCHAR(15) PRIMARY KEY,        -- Play identifier (VARCHAR), NOT NULL (Primary Key)
        game_id VARCHAR(12) NOT NULL,           -- Game identifier (INT), NOT NULL (every play is linked to a game)
        team_id_for VARCHAR(7),                 -- Team initiating the play (INT), can be NULL if unknown
        team_id_against VARCHAR(7),             -- Opposing team for the play (INT), can be NULL if unknown
        event VARCHAR(25),                      -- Event type (VARCHAR), can be NULL if no event
        secondary_type VARCHAR(40),             -- Secondary event type (VARCHAR), can be NULL if missing
        x INT,                                  -- x-coordinate of the play (FLOAT), can be NULL if missing
        y INT,                                  -- y-coordinate of the play (FLOAT), can be NULL if missing
        period INT,                             -- Period of the game (INT), can be NULL if not applicable
        period_type VARCHAR(20),                -- Period type (VARCHAR), can be NULL if unknown
        period_time INT,                        -- Time of the play in the period (INT), can be NULL if missing
        period_time_remaining INT,              -- Remaining time in the period (FLOAT), can be NULL if missing
        date_time TIMESTAMP,
        goals_away INT,                         -- Goals by away team (INT), can be NULL if not applicable
        goals_home INT,                         -- Goals by home team (INT), can be NULL if not applicable
        description VARCHAR(255),                       -- Description of the play (TEXT), can be NULL if missing
        st_x INT,                               -- Start x-coordinate of the play (FLOAT), can be NULL if missing
        st_y INT                               -- Start y-coordinate of the play (FLOAT), can be NULL if missing
        
    );
    """,
       
    # Table 5: game_plays_players
    """
    CREATE TABLE game_plays_players (
        plays_players_id SERIAL PRIMARY KEY,
        play_id VARCHAR(15) NOT NULL,
        game_id VARCHAR(12) NOT NULL,
        player_id VARCHAR(10) NOT NULL,         
        player_type VARCHAR(50)
        
    );
    """,
 
     
    # Table 6: game_goals
    """
    CREATE TABLE game_goals (
        goals_id SERIAL PRIMARY KEY,
        play_id VARCHAR(15),
        strength VARCHAR(20),  
        game_winning_goal VARCHAR(5),
        empty_net VARCHAR(5)
    );
    """,
   
      # Table 4: game_goalie_stats
    """
    CREATE TABLE game_goalie_stats (
        goalie_stats_id SERIAL PRIMARY KEY,
        game_id VARCHAR(12) NOT NULL,
        player_id VARCHAR(10),
        team_id VARCHAR(3),
        time_on_ice INT,
        assists INT ,
        goals INT ,
        penalty_minutes INT, -- Penalty in minutes
        shots INT,
        saves INT,
        power_play_saves INT ,
        short_handed_saves INT ,
        even_saves INT ,
        short_handed_shots_against INT ,
        even_shots_against INT ,
        power_play_shots_against INT ,
        decision CHAR(5), -- 'W' (Win) or 'L' (Loss)
        save_percentage FLOAT,
        power_play_save_percentage FLOAT,
        even_strength_save_percentage FLOAT
        
    );
    """,
    
    # Table 8 : game_skater_stats
    """
    CREATE TABLE game_skater_stats (
        skater_stats_id SERIAL PRIMARY KEY,
        game_id VARCHAR(12),
        player_id VARCHAR(10),
        team_id VARCHAR(3),
        time_on_ice INT,
        assists INT,
        goals INT,
        shots INT,
        hits FLOAT,  -- Nullable column
        power_play_goals INT,
        power_play_assists INT,
        penalty_minutes INT,
        face_off_wins INT,
        face_off_taken INT,
        takeaways FLOAT,  -- Nullable column
        giveaways FLOAT,  -- Nullable column
        short_handed_goals INT,
        short_handed_assists INT,
        blocked FLOAT,  -- Nullable column
        plus_minus INT,
        even_time_on_ice INT,
        short_handed_time_on_ice INT,
        power_play_time_on_ice INT
   
    );
    """,
     
    # Table 9 : game_team_stats
    """
    CREATE TABLE game_team_stats (
        team_stats_id SERIAL PRIMARY KEY,               ---- Surrogate primary key
        game_id VARCHAR(12) NOT NULL,
        team_id VARCHAR(3) NOT NULL,
        hoa VARCHAR(4) , 
        won VARCHAR(5) ,                        -- 'Yes' or 'No'
        settled_in VARCHAR(3) NOT NULL,              -- Example: "REG", "OT", "SO"
        head_coach VARCHAR(50),
        goals INT,
        shots INT,
        hits FLOAT,
        penalty_minutes INT,                           -- Total penalty minutes for the team.
        power_play_opps INT,                        -- Number of power-play opportunities for the team.
        power_play_goals INT,                       -- Goals scored during power plays.
        face_off_win_percentage FLOAT,      -- Percentage of faceoffs won by the team.
        giveaways FLOAT,
        takeaways FLOAT,
        blocked FLOAT,
        start_rink_side VARCHAR(7)                 -- 'left' or 'right'
    
      );
      """,

    # Table 10 : game_penalties
    """ 
    CREATE TABLE game_penalties (
        penalties_id SERIAL PRIMARY KEY,        -- Surrogate primary key
        play_id VARCHAR(15) NOT NULL,   -- Foreign key referencing game_plays
        penalty_severity VARCHAR(20), -- Penalty severity (e.g., minor, major)
        penalty_minutes INT          -- Duration of the penalty in minutes
   );     
    """,
    
    # Table 11 : game_officials
    """ 
    CREATE TABLE game_officials (
        officials_id SERIAL PRIMARY KEY,     --- Surrogate primary key
        game_id VARCHAR(12),
        official_name VARCHAR(50) ,
        official_type VARCHAR(20) 
       
    );
    """,
    
   
    # Table 12 : game_shifts
    """ 
    CREATE TABLE game_shifts (
        shifts_id SERIAL PRIMARY KEY,        --- Surrogate primary key
        game_id VARCHAR(12) ,
        player_id VARCHAR(10) ,
        period INT,
        shift_start INT,
        shift_end FLOAT
    
    ); 
    """,   
  
  # Table 13 : game_scratches
    """ 
    CREATE TABLE game_scratches (
        scratches_id SERIAL PRIMARY KEY, 
        game_id VARCHAR(12),
        team_id VARCHAR(3),
        player_id VARCHAR(10)
    
    );
    """,
]
try:
    with engine.connect() as connection:
        for query in commands:
            connection.execute(text(query))
        connection.commit()  # Explicit commit
    print("Tables created successfully.")
except SQLAlchemyError as e:
    print(f"Error occurred: {e}")


Tables created successfully.


### Upload Dataframe to PostgreSQL
Define your file_path

e.g. if your file path for team_info.csv is 'C:\Users\NHL\team_info.csv'

file_path = r'C:\Users\NHL\team_info.csv'

In [6]:
# Function to upload a DataFrame to PostgreSQL
def upload_to_postgres(df, table_name):
    try:
        # Upload the DataFrame to the specified table
        df.to_sql(table_name, engine, index=False, if_exists='append')
        print(f"Data has been successfully uploaded to the table '{table_name}'.")
    except SQLAlchemyError as e:
        print(f"Error uploading to the table '{table_name}': {e}")

# Manually specify and load each file into a DataFrame, then upload it to PostgreSQL

# File 1: team_info
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\team_info.csv"
df_team_info = pd.read_csv(file_path)
print(f"Loaded team_info.csv with {len(df_team_info)} rows.")
upload_to_postgres(df_team_info, "team_info")

# File 2: player_info
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\player_info.csv"
df_player_info = pd.read_csv(file_path, parse_dates=["birth_date"])  # parse as dates
print(f"Loaded player_info.csv with {len(df_player_info)} rows.")
upload_to_postgres(df_player_info, "player_info")

# File 3: game
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game.csv"
df_game = pd.read_csv(file_path, parse_dates=["date_time_gmt"])
print(f"Loaded game.csv with {len(df_game)} rows.")
upload_to_postgres(df_game, "game")

# File 4: game_plays
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_plays.csv"
df_game_plays = pd.read_csv(file_path, dtype={
        "team_id_for": "string",  # set to string
        "team_id_against": "string",  # set to string
         },
        parse_dates=["date_time"]  # parse as dates
                           )                    
print(f"Loaded game_plays.csv with {len(df_game_plays)} rows.")
upload_to_postgres(df_game_plays, "game_plays")

# File 5: game_plays_players
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_plays_players.csv"
df_game_plays_players = pd.read_csv(file_path)
print(f"Loaded game_plays_players.csv with {len(df_game_plays_players)} rows.")
upload_to_postgres(df_game_plays_players, "game_plays_players")

# File 6: game_goals
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_goals.csv"
df_game_goals = pd.read_csv(file_path)
print(f"Loaded game_goals.csv with {len(df_game_goals)} rows.")
upload_to_postgres(df_game_goals, "game_goals")

# File 7: game_goalie_stats
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_goalie_stats.csv"
df_game_goalie_stats = pd.read_csv(file_path)
print(f"Loaded game_goalie_stats.csv with {len(df_game_goalie_stats)} rows.")
upload_to_postgres(df_game_goalie_stats, "game_goalie_stats")

# File 8: game_skater_stats
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_skater_stats.csv"
df_game_skater_stats = pd.read_csv(file_path)
print(f"Loaded game_skater_stats.csv with {len(df_game_skater_stats)} rows.")
upload_to_postgres(df_game_skater_stats, "game_skater_stats")

# File 9: game_team_stats
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_team_stats.csv"
df_game_team_stats = pd.read_csv(file_path)
print(f"Loaded game_team_stats.csv with {len(df_game_team_stats)} rows.")
upload_to_postgres(df_game_team_stats, "game_team_stats")

# File 10: game_penalties
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_penalties.csv"
df_game_penalties = pd.read_csv(file_path)
print(f"Loaded game_penalties.csv with {len(df_game_penalties)} rows.")
upload_to_postgres(df_game_penalties, "game_penalties")

# File 11: game_officials
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_officials.csv"
df_game_officials = pd.read_csv(file_path)
print(f"Loaded game_officials.csv with {len(df_game_officials)} rows.")
upload_to_postgres(df_game_officials, "game_officials")

# File 12: game_shifts
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_shifts.csv"
df_game_shifts = pd.read_csv(file_path)
print(f"Loaded game_shifts.csv with {len(df_game_shifts)} rows.")
upload_to_postgres(df_game_shifts, "game_shifts")

# File 13: game_scratches
file_path = r"C:\Users\zacle\Desktop\Serene\Project\NHL\ETL\clean\game_scratches.csv"
df_game_scratches = pd.read_csv(file_path)
print(f"Loaded game_scratches.csv with {len(df_game_scratches)} rows.")
upload_to_postgres(df_game_scratches, "game_scratches")

Loaded team_info.csv with 33 rows.
Data has been successfully uploaded to the table 'team_info'.
Loaded player_info.csv with 3925 rows.
Data has been successfully uploaded to the table 'player_info'.
Loaded game.csv with 23735 rows.
Data has been successfully uploaded to the table 'game'.
Loaded game_plays.csv with 4217063 rows.
Data has been successfully uploaded to the table 'game_plays'.
Loaded game_plays_players.csv with 6362801 rows.
Data has been successfully uploaded to the table 'game_plays_players'.
Loaded game_goals.csv with 131501 rows.
Data has been successfully uploaded to the table 'game_goals'.
Loaded game_goalie_stats.csv with 51125 rows.
Data has been successfully uploaded to the table 'game_goalie_stats'.
Loaded game_skater_stats.csv with 853404 rows.
Data has been successfully uploaded to the table 'game_skater_stats'.
Loaded game_team_stats.csv with 47442 rows.
Data has been successfully uploaded to the table 'game_team_stats'.
Loaded game_penalties.csv with 229228 